# 🐄 Cattle Analytics — Colab Pro+ Experiment Notebook

**Pipeline**: YOLOv8 Detection → ByteTrack Tracking → MobileNetV3 Behavior Classification

**Data layout on Drive**:
```
MyDrive/cattle-analytics/data/
├── v1/  ← video + annotations
│   ├── <video>.mp4
│   └── annotations.xml
├── v2/ ... vN/  ← add new versions here as you annotate
```

**Runtime**: Runtime → Change runtime type → **A100 GPU** (Colab Pro+)

Scripts 01, 02, 03 are **incremental** — they skip already-processed versions automatically.

## 📦 Cell 1 — Install Dependencies
> Run once per session. Subsequent runs are fast (cached by pip).

In [ ]:
%%capture
!pip install ultralytics==8.3.0
!pip install supervision==0.21.0
!pip install lap
!pip install albumentations
!pip install timm
!pip install seaborn scikit-learn tabulate

import torch
print(f'CUDA available : {torch.cuda.is_available()}')
print(f'GPU            : {torch.cuda.get_device_name(0)}')
print(f'VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## ☁️ Cell 2 — Mount Drive & Pull Code
> Clones your experiment repo from GitHub (first time) or pulls updates.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys

# ── CONFIGURE THESE ─────────────────────────────────
DRIVE_ROOT  = '/content/drive/MyDrive/cattle-analytics'
GITHUB_REPO = 'https://github.com/YOUR_ORG/cattle-analytics.git'
CODE_DIR    = f'{DRIVE_ROOT}/cattle-tracker'
# ────────────────────────────────────────────────

os.makedirs(DRIVE_ROOT, exist_ok=True)

if os.path.exists(f'{DRIVE_ROOT}/.git'):
    print('Repo already cloned. Pulling latest...')
    !cd {DRIVE_ROOT} && git pull
else:
    print('Cloning repo...')
    !git clone {GITHUB_REPO} {DRIVE_ROOT}

sys.path.insert(0, CODE_DIR)
os.chdir(CODE_DIR)
print(f'Working dir: {os.getcwd()}')

## 🗂️ Cell 3 — Setup Drive Paths & Discover Version Folders
> Mounts Drive paths + discovers all version folders (`v1`, `v2`, … `vN`).  
> Each folder needs exactly one `.mp4` + one `annotations.xml` to be included.

In [ ]:
import scripts.setup_drive as cfg

VERSION_FOLDERS = cfg.VERSION_FOLDERS
FRAMES_DIR      = cfg.FRAMES_DIR
YOLO_DIR        = cfg.YOLO_DIR
CROPS_DIR       = cfg.CROPS_DIR
MODELS_DIR      = cfg.MODELS_DIR
LOGS_DIR        = cfg.LOGS_DIR

print(f'Discovered {len(VERSION_FOLDERS)} version folder(s): {[v.name for v in VERSION_FOLDERS]}')

## 🏞️ Cell 4 — Extract Annotated Frames *(incremental)*
> Reads each XML → finds annotated frame IDs → extracts + resizes to 1280×720.  
> **Incremental**: writes a `.done` sentinel after each version.  
> Already-done versions are skipped — only new versions are processed.  
> Output: `processed/frames/vX/<stem>/frame_XXXXXX.jpg`

In [ ]:
!python scripts/01_extract_frames.py --width 1280 --height 720

# Force re-extract specific versions (e.g. if you fixed annotations):
# !python scripts/01_extract_frames.py --force v13 v14
#
# Force re-extract everything from scratch:
# !python scripts/01_extract_frames.py --force

## 🏷️ Cell 5 — Convert CVAT XML → YOLO Format *(incremental)*
> Creates YOLO labels + `dataset.yaml`. Split is done at **video level** (no temporal leakage).  
> **Incremental**: already-processed versions are skipped; their cached stats  
> are merged into `dataset_stats.json` so it always reflects the full dataset.  
> Already-done versions **keep their original split — never changed**.

In [ ]:
# Edit VAL_VERSIONS: space-separated version names that should be validation set.
# Only applies to NEW versions — already-processed versions keep their existing split.
VAL_VERSIONS = 'v7'   # e.g. 'v7 v14 v21' for multiple

!python scripts/02_cvat_to_yolo.py \
    --split_mode manual \
    --val_versions {VAL_VERSIONS}

# Auto mode (picks val versions by frame count to hit ~20%):
# !python scripts/02_cvat_to_yolo.py --split_mode auto --val_split 0.2
#
# Force re-convert specific versions:
# !python scripts/02_cvat_to_yolo.py --val_versions {VAL_VERSIONS} --force v13

## ✂️ Cell 6 — Generate Behavior Crops *(incremental)*
> Crops each annotated calf per behavior → `processed/behavior_crops/<behavior>/`  
> Adds 15% padding around bounding boxes for context.  
> **Incremental**: writes `.done_vX.json` per version; crop counts cached and merged.

In [ ]:
!python scripts/03_generate_crops.py --padding 0.15

# Force re-crop specific versions:
# !python scripts/03_generate_crops.py --force v13 v14

## 🔍 Cell 7 — Dataset Health Check
> **Always run this before training.**  
> Shows frame counts per split, behavior distribution, class imbalance warnings,  
> missing label files, and crop counts per behavior class.

In [ ]:
!python scripts/04_check_dataset_stats.py

## 🖼️ Cell 8 — Visualize Annotations *(optional sanity check)*
> Draws bounding boxes + behavior labels onto sampled frames.  
> Great for catching annotation errors before training.

In [ ]:
!python scripts/05_visualize_annotations.py --version v1 --max_frames 10

from IPython.display import Image as IPyImage
import glob
for s in glob.glob('processed/viz/v1/*.jpg')[:3]:
    display(IPyImage(s, width=800))

## 🏷️ Cell 9 — Train YOLOv8 Calf Detector
> Trains YOLOv8s on your calf detection dataset.  
> Metrics logged to `logs/experiments/detector_<ts>/metrics.json`  
> Weights saved to `models/checkpoints/detector_<ts>/weights/best.pt`  
> ⏱️ ~30–60 min on A100 for 100 epochs.  
> **Tip**: Run 30 epochs first as a smoke test.

In [ ]:
# Smoke test — run first to confirm pipeline works
!python train/train_detector.py \
    --model yolov8s.pt \
    --epochs 30 \
    --batch 8 \
    --imgsz 1280

In [ ]:
# Full training run — uncomment when satisfied with smoke test
# !python train/train_detector.py \
#     --model yolov8s.pt \
#     --epochs 100 \
#     --batch 8 \
#     --imgsz 1280

## 🧠 Cell 10 — Train Behavior Classifier
> Trains MobileNetV3-Small on behavior crops.  
> Uses WeightedRandomSampler + class-weighted loss to handle imbalance.  
> `num_workers=0` is required on Colab (Drive-backed loaders deadlock with workers > 0).  
> ⚠️ **Prerequisite**: Need ≥200 crops per class — check Cell 7 output first.  
> ⏱️ ~10–20 min on A100 for 50 epochs.

In [ ]:
!python train/train_classifier.py \
    --epochs 50 \
    --batch 64 \
    --lr 1e-4

## 📊 Cell 11 — Evaluate & Compare All Experiments
> Confusion matrix + per-class report + experiment comparison table.

In [ ]:
!python evaluation/eval_pipeline.py --phase summary

In [ ]:
!python evaluation/eval_pipeline.py --phase classifier

from IPython.display import Image as IPyImage
import glob
cms = sorted(glob.glob('models/checkpoints/classifier_*/confusion_matrix.png'))
if cms:
    display(IPyImage(cms[-1], width=700))

## 🎦 Cell 12 — Run Full Inference Pipeline on a Video
> Detect → Track → Classify → Annotated video + JSON output

In [ ]:
import glob, os

detectors   = sorted(glob.glob('models/checkpoints/detector_*/weights/best.pt'))
classifiers = sorted(glob.glob('models/checkpoints/classifier_*/best_classifier.pth'))

DETECTOR_PATH   = detectors[-1]   if detectors   else None
CLASSIFIER_PATH = classifiers[-1] if classifiers else None

INPUT_VIDEO  = 'data/v1/<your_video>.mp4'   # UPDATE THIS
OUTPUT_VIDEO = 'data/v1/output_annotated.mp4'

print(f'Detector   : {DETECTOR_PATH}')
print(f'Classifier : {CLASSIFIER_PATH}')
print(f'Input      : {INPUT_VIDEO}')

if DETECTOR_PATH and CLASSIFIER_PATH and os.path.exists(INPUT_VIDEO):
    !python inference/pipeline.py \
        --video        {INPUT_VIDEO} \
        --detector     {DETECTOR_PATH} \
        --classifier   {CLASSIFIER_PATH} \
        --output_video {OUTPUT_VIDEO} \
        --conf 0.35 \
        --classify_every 4
else:
    print('Missing detector, classifier, or input video. Check paths above.')

## ➕ Cell 13 — Add More Videos & Retrain
> **Steps to add new videos**:
> 1. Upload new video+XML to `data/vN/` on Drive (e.g. `data/v35/`)
> 2. Re-run **Cells 3 → 11** — v1–v34 are skipped, only vN is processed
> 3. Cell 11 automatically compares the new run against all previous ones
>
> **Split assignment for new videos**:  
> Edit `VAL_VERSIONS` in Cell 5 to add new val versions.  
> Already-processed versions **keep their original split — never changed**.

In [ ]:
# Re-discover version folders after uploading new videos to Drive
import importlib
import scripts.setup_drive as cfg
importlib.reload(cfg)

VERSION_FOLDERS = cfg.VERSION_FOLDERS
print(f'Total version folders: {len(VERSION_FOLDERS)}')
print(f'Folders: {[v.name for v in VERSION_FOLDERS]}')

# Show which have already been extracted
done    = [v.name for v in VERSION_FOLDERS if (cfg.FRAMES_DIR / v.name / '.done').exists()]
pending = [v.name for v in VERSION_FOLDERS if v.name not in done]
print(f'Already extracted : {done}')
print(f'Pending extraction: {pending}')

In [ ]:
# Force re-process specific versions (e.g. if you fixed annotations)
# !python scripts/01_extract_frames.py --force v13 v14
# !python scripts/02_cvat_to_yolo.py   --force v13 v14
# !python scripts/03_generate_crops.py --force v13 v14

# Force re-process everything from scratch (nuclear option)
# !python scripts/01_extract_frames.py --force
# !python scripts/02_cvat_to_yolo.py   --force
# !python scripts/03_generate_crops.py --force